In [ ]:
import sys
from pathlib import Path

# Ensure project-root imports (e.g., utils.*) work regardless notebook launch directory.
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'utils').is_dir():
        if str(candidate) not in sys.path:
            sys.path.append(str(candidate))
        break

import numpy as np
import pandas as pd
import pickle
import os
import math
import seaborn as sns
import matplotlib as mpl
mpl.rcParams['legend.markerscale'] = 1.5
mpl.rcParams['pdf.fonttype'] = 42
import matplotlib.pyplot as plt
from tqdm import tqdm
import ot
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
import warnings
from sklearn.metrics import pairwise_distances, silhouette_score, v_measure_score, calinski_harabasz_score, adjusted_rand_score, davies_bouldin_score
from sklearn.cluster import KMeans, AgglomerativeClustering, HDBSCAN
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
import umap
from PIL import Image
from utils.config import load_env_paths
from utils.mypalette import mutation_palette, condition_palette, cell_line_palette, palette_mutcond
from utils.snn import *
from utils.start_notebook import setup_experiment
from sklearnex import patch_sklearn
patch_sklearn()

warnings.filterwarnings('ignore')

ENV_PATHS = load_env_paths(str((candidate / 'configs' / 'env.yml').resolve()))

x = '240813_5'
single_exp = ['E1']
cluster_0 = False
emb_type = 'deep'
assert emb_type in ['deep', 'handcrafted']

# Choose which metric to optimize when selecting the clustering configuration.
selection_metric = 'ARI'  # or 'silhouette'
selection_method = None   # None, 'KMeans', or 'SNN'
assert selection_metric in ['ARI', 'silhouette']

saving_dir = os.path.join(ENV_PATHS['temp_results_root_dir'], x, 'clustering', ('_').join(single_exp))
os.makedirs(saving_dir, exist_ok=True)

annotations_df, raw_emb = setup_experiment(x, single_exp)

cluster_metric_path = os.path.join(ENV_PATHS['temp_results_root_dir'], x, 'clustering', 'UMAP', f'clustering_algo_comparison_{("_").join(single_exp)}.csv')

if not os.path.exists(cluster_metric_path):
    raise FileNotFoundError(f'Could not find clustering metrics file: {cluster_metric_path}')

cluster_evaluations = pd.read_csv(cluster_metric_path)

categorical_labels = annotations_df['mut_cond'].values
embeddings = {}

# Specify the desired order for mut_cond
desired_order = ['CTRL_untreated', 'CTRL_A1 low', 'CTRL_A1 high',
                 'VCP_untreated', 'VCP_A1 low', 'VCP_A1 high']
mut_order = ['CTRL', 'VCP']
cond_order = ['untreated', 'A1 low', 'A1 high']

if 'A1 low' not in annotations_df.condition.values:
    cond_order.remove('A1 low')
    desired_order = [cls for cls in desired_order if 'A1 low' not in cls]

# Remove classes from desired_order that are not present in the mut_cond column
desired_order = [cls for cls in desired_order if cls in annotations_df['mut_cond'].unique()]
# Convert mut_cond to a categorical type with the specified order
annotations_df['mut_cond'] = pd.Categorical(annotations_df['mut_cond'], categories=desired_order, ordered=True)
annotations_df['mut_cond'] = annotations_df['mut_cond'].apply(lambda x: x.replace('_', '\n'))
annotations_df['cellline_cond'] = annotations_df['cell_line'].astype(str) + '_' + annotations_df['condition'].astype(str)

dataset_size = len(annotations_df)

# Prefer compare_different_clustering.py plot-eligible rows when available.
if 'include_for_plots' in cluster_evaluations.columns:
    cluster_evaluations = cluster_evaluations[cluster_evaluations['include_for_plots'] == True]
else:
    cluster_evaluations = cluster_evaluations[cluster_evaluations['unassigned'] <= 0.15 * dataset_size]

cluster_evaluations = cluster_evaluations[cluster_evaluations['n_clusters'] >= 2]
cluster_evaluations = cluster_evaluations[cluster_evaluations['method'].isin(['SNN', 'KMeans'])]

if selection_method is not None:
    cluster_evaluations = cluster_evaluations[cluster_evaluations['method'] == selection_method]

if cluster_evaluations.empty:
    raise ValueError('No clustering configurations left after filtering.')

if selection_metric not in cluster_evaluations.columns:
    raise ValueError(f'Metric {selection_metric} not found in clustering metrics file.')

# Compute/retrieve best clustering labels

In [ ]:
# Compute only the UMAP embeddings that are required by the selected metrics table.
required_umap_neighbors = sorted({
    int(str(name).split('_')[-1])
    for name in cluster_evaluations['embedding'].dropna().unique()
    if str(name).startswith('umap_n_neighbors_')
})

if len(required_umap_neighbors) == 0:
    required_umap_neighbors = [15]

print(f'Required UMAP neighbors from metrics file: {required_umap_neighbors}')

for n_neighbors in required_umap_neighbors:
    umap_emb = umap.UMAP(n_neighbors=n_neighbors, random_state=0, n_components=2, n_jobs=1).fit_transform(raw_emb)
    embeddings[f'umap_n_neighbors_{n_neighbors}'] = umap_emb

In [ ]:
def apply_knn_to_unassigned(emb, labels, k=25):
    labels = np.asarray(labels).copy()
    unassigned_indices = np.where(labels == -1)[0]
    if len(unassigned_indices) == 0:
        return labels

    assigned_mask = labels != -1
    if np.sum(assigned_mask) == 0:
        return labels

    n_neighbors = min(k, int(np.sum(assigned_mask)))
    nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(emb[assigned_mask])
    _, indices = nbrs.kneighbors(emb[unassigned_indices])

    assigned_labels = labels[assigned_mask]
    for idx, neighbors in zip(unassigned_indices, indices):
        neighbor_labels = assigned_labels[neighbors]
        majority_label = np.bincount(neighbor_labels).argmax()
        labels[idx] = majority_label

    return labels

# Select best configuration by metric (highest ARI or highest silhouette).
best_row = cluster_evaluations.sort_values(by=selection_metric, ascending=False).iloc[0]
print('Selected configuration:')
print(best_row[['method', 'embedding', 'n_clusters', selection_metric]])

embedding_name = str(best_row['embedding'])
if embedding_name not in embeddings:
    raise ValueError(f'Embedding {embedding_name} was not computed.')
emb_to_analyze = embeddings[embedding_name]

legend_order = ['CTRL1', 'CTRL2', 'CTRL4', 'CTRL5', 'GLIA', 'GLIB', 'CBI', 'CBID', 'CBIE']

method_name = str(best_row['method'])
if method_name == 'KMeans':
    cl_method = KMeans(n_clusters=int(best_row['n_clusters']), random_state=0)
elif method_name == 'SNN':
    cl_method = SNN(
        neighbor_num=int(best_row['neighbor_num']),
        min_shared_neighbor_proportion=float(best_row['min_shared_neighbor_proportion']),
        clustering_algo='DBSCAN'
    )
else:
    raise ValueError(f'Unsupported method in selected row: {method_name}')

best_labels = cl_method.fit_predict(emb_to_analyze)

# If compare_different_clustering.py applied KNN correction for this row, reproduce it here.
knn_applied_flag = False
if 'knn_applied' in cluster_evaluations.columns:
    knn_applied_flag = bool(best_row['knn_applied'])

if knn_applied_flag:
    best_labels = apply_knn_to_unassigned(emb_to_analyze, best_labels, k=25)

if not cluster_0 and min(best_labels) == 0:
    best_labels = best_labels + 1
annotations_df['cluster'] = best_labels

num_clusters = len(np.unique(best_labels))
if num_clusters > 10:
    cl_palette = sns.color_palette('tab20', 20) + sns.color_palette('husl', num_clusters - 20)
else:
    cl_palette = sns.color_palette('tab10', num_clusters)

fig, axes = plt.subplots(1, 4, figsize=(32, 8))
for i, (label, palette) in enumerate(zip(['cell_line', 'mutation', 'condition', 'cluster'], [cell_line_palette, mutation_palette, condition_palette, cl_palette])):
    hue_data = annotations_df[label] if label != 'cluster' else best_labels
    sns.scatterplot(x=emb_to_analyze[:, 0],
                    y=emb_to_analyze[:, 1],
                    hue=hue_data,
                    palette=palette, ax=axes[i],
                    s=12, alpha=0.5)

    axes[i].set_title(label)
    axes[i].legend(markerscale=2, fancybox=True, loc='best')
    if label == 'cell_line':
        handles, labels = axes[i].get_legend_handles_labels()
        ordered_handles_labels = sorted(zip(handles, labels), key=lambda x: legend_order.index(x[1]) if x[1] in legend_order else float('inf'))
        ordered_handles, ordered_labels = zip(*ordered_handles_labels)
    for lh in axes[i].get_legend().legend_handles:
        lh.set_alpha(1)

plt.show()

In [ ]:
change_labels = True

change_dict = {
        1: 1, 
        2: 2,  
        3: 5,
        4: 6, 
        5: 3,
        6: 4,
}

#annotations_df['cluster'] = annotations_df['cluster'].astype('category')


if change_labels:
    # Use .map() or .replace() to safely swap the values in the column
    annotations_df['cluster'] = annotations_df['cluster'].map(change_dict)

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for i, (label, palette) in enumerate(zip(['mutation', 'condition', 'cluster'], [mutation_palette, condition_palette, cl_palette])):
    hue_data = annotations_df[label]
    sns.scatterplot(x=emb_to_analyze[:, 0],
                    y=emb_to_analyze[:, 1],
                    hue=hue_data,
                    palette=palette, ax=axes[i],
                    s=12, alpha=0.5)
        
    axes[i].set_title(label)
    axes[i].legend(markerscale=2, fancybox=True, loc='best')
    for lh in axes[i].get_legend().legend_handles:
        lh.set_alpha(1)

## number of images per cluster

In [ ]:
# count number of images per cluster

# Assuming 'cluster_labels' is a list or array of cluster assignments for each image
cluster_counts = pd.Series(best_labels).value_counts()
print(cluster_counts)
# color bars according to cluster label using cl_palette

plt.subplots(figsize=(8, 8))
sns.barplot(x=cluster_counts.index, y=cluster_counts.values, palette=cl_palette)
plt.savefig(os.path.join(saving_dir, "cluster_counts.pdf"), dpi=300)
plt.show()

## cluster prototypes

In [ ]:
centroids = {i: emb_to_analyze[best_labels == i].mean(axis=0) for i in np.unique(best_labels)}

n_closest = 10
img_size = (64, 64)  # Adjust if your images are a different size
num_clusters = len(centroids)

fig, axes = plt.subplots(num_clusters, n_closest, figsize=(n_closest * 2, num_clusters * 2))
if num_clusters == 1:
    axes = np.expand_dims(axes, 0)  # Ensure axes is 2D

for row_idx, (i, cl_center) in enumerate(centroids.items()):
    temp_df = annotations_df[annotations_df['cluster'] == i].copy()
    temp_emb = emb_to_analyze[best_labels == i]
    # Compute distances and keep original indices
    distances = pd.DataFrame(pairwise_distances(temp_emb, [cl_center]), columns=['distance'])
    distances['orig_index'] = temp_df.index.to_numpy()
    # Sort by distance
    distances = distances.sort_values(by='distance', ascending=True).reset_index(drop=True)
    # Add img_code in the correct order
    distances['img_code'] = annotations_df.loc[distances['orig_index'], 'img_code'].values
    distances = distances.drop(columns='orig_index')
    
    # Select the 10 closest images
    closest_imgs = distances['img_code'].iloc[:n_closest]
    
    for col_idx, img_path in enumerate(closest_imgs):
        ax = axes[row_idx, col_idx] if num_clusters > 1 else axes[0, col_idx]
        try:
            img = Image.open(img_path)
            ax.imshow(img)
            ax.axis('off')
        except Exception:
            ax.axis('off')
    # Hide any unused axes in the row
    for col_idx in range(len(closest_imgs), n_closest):
        ax = axes[row_idx, col_idx] if num_clusters > 1 else axes[0, col_idx]
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(saving_dir, 'closest_images_grid.pdf'), format='PDF', dpi=300)
plt.show()

In [ ]:
# Calculate the proportion of classes in each cluster
class_proportion = annotations_df.groupby(['cluster', 'mut_cond']).size().unstack(fill_value=0)
class_proportion = class_proportion.div(class_proportion.sum(axis=1), axis=0)
adjustment = 0 if cluster_0 else 1

# Define labels for the legend
labels = [f'Cluster {i+adjustment}' for i in range(len(class_proportion))]

## mutation, condition piecharts (one for each cluster)

In [ ]:
# For each cluster, display a pie chart of its composition with respect to mut_cond
clusters = sorted(annotations_df['cluster'].unique())
mut_cond_labels = annotations_df['mut_cond'].cat.categories if hasattr(annotations_df['mut_cond'], 'cat') else sorted(annotations_df['mut_cond'].unique())
n_clusters = len(clusters)
ncols = 3
nrows = (n_clusters // ncols) + (n_clusters % ncols > 0)

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
axes = axes.flatten() if n_clusters > 1 else [axes]

for idx, cluster_label in enumerate(clusters):
    #fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    sub = annotations_df[annotations_df['cluster'] == cluster_label]
    counts = sub['mut_cond'].value_counts().reindex(mut_cond_labels, fill_value=0)
    proportions = counts / counts.sum() if counts.sum() > 0 else counts
    colors = [palette_mutcond.get(mc.replace('\n', '_'), '#888888') for mc in mut_cond_labels]
    axes[idx].pie(proportions, autopct=None, colors=colors, startangle=140)
    axes[idx].set_title(f'Cluster {cluster_label}', fontsize=14)

# add a legend based on mut_cond_labels and palette_mutcond
handles = []
labels = []
for mc, color in zip(mut_cond_labels, colors):
    handles.append(plt.Rectangle((0,0),1,1, color=color))
    labels.append(mc)

axes[0].legend(handles, labels, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'mutcond_piecharts_{("_").join(single_exp)}.pdf'), dpi=300)
plt.show()

## cell line piecharts (for each cluster)

In [ ]:
# For each cluster, display a pie chart of its composition with respect to mut_cond
clusters = sorted(annotations_df['cluster'].unique())
line_labels = annotations_df['cell_line'].cat.categories if hasattr(annotations_df['cell_line'], 'cat') else sorted(annotations_df['cell_line'].unique())
n_clusters = len(clusters)
ncols = 3
nrows = (n_clusters // ncols) + (n_clusters % ncols > 0)

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
axes = axes.flatten() if n_clusters > 1 else [axes]

for idx, cluster_label in enumerate(clusters):
    #fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    sub = annotations_df[annotations_df['cluster'] == cluster_label]
    counts = sub['cell_line'].value_counts().reindex(line_labels, fill_value=0)
    proportions = counts / counts.sum() if counts.sum() > 0 else counts
    colors = [cell_line_palette.get(mc.replace('\n', '_'), '#888888') for mc in line_labels]
    axes[idx].pie(proportions, autopct=None, colors=colors, startangle=140)
    axes[idx].set_title(f'Cluster {cluster_label}', fontsize=14)

legend_order = ['CTRL1', 'CTRL2', 'CTRL5', 'GLIA', 'GLIB', 'CBID', 'CBIE']
# add a legend based on line_labels ordered according to legend order
handles = []
labels = []
for label in legend_order:
    # Check if this specific label exists in your current data map
    if label in cell_line_palette:
        # Create the handle using the mapped color
        handles.append(plt.Rectangle((0,0), 1, 1, color=cell_line_palette[label]))
        labels.append(label)

axes[0].legend(handles, labels, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'cellline_piecharts_{("_").join(single_exp)}.pdf'), dpi=300)
plt.show()

## Dendrogram for clusters

In [ ]:
global_labels = annotations_df['cluster'].unique()
wasserstein_pairwise = np.zeros((len(global_labels), len(global_labels)))
for i, label_1 in enumerate(global_labels):
    for j, label_2 in enumerate(global_labels):
        if i < j:  # Compute only for upper triangle
            pts1 = raw_emb[annotations_df['cluster'] == label_1]
            pts2 = raw_emb[annotations_df['cluster'] == label_2]
            wasserstein_pairwise[i, j] = ot.emd2(ot.unif(len(pts1)), ot.unif(len(pts2)), ot.dist(pts1, pts2))
            wasserstein_pairwise[j, i] = wasserstein_pairwise[i, j]  # Symmetric matrix
# Perform hierarchical clustering
linkage_matrix = linkage(squareform(wasserstein_pairwise), method='ward')
# Plot dendrogram
fig, axes = plt.subplots(1, 1, figsize=(8, 6))

# Dendrogram colored by mutation
dendro = dendrogram(
    linkage_matrix,
    labels=global_labels,
    leaf_rotation=0,
    leaf_font_size=10,
    orientation='top',
    link_color_func=lambda k: 'black',
    ax=axes
)
ordered_clusters = [int(lbl) for lbl in dendro['ivl']]

cl_palette_dict = {str(i+1): cl_palette[i] for i in range(len(cl_palette))}

for label in axes.get_xticklabels():
    text = label.get_text()
    label.set_color(cl_palette_dict.get(text, 'black'))
#axes.set_title('Colored by Cluster')
axes.set_ylabel('Wasserstein Distance')
axes.set_xlabel('Cluster')
plt.savefig(os.path.join(saving_dir, f'dendrogram_clusters_{("_").join(single_exp)}.pdf'), dpi=300)

## Enrichment analysis

In [ ]:
from scipy.stats import chi2_contingency, norm

tab = pd.crosstab(annotations_df['cluster'], annotations_df['mut_cond'])
chi2, p_asymp, dof, expected = chi2_contingency(tab.values, correction=False)

expected_df = pd.DataFrame(expected, index=tab.index, columns=tab.columns)
min_exp = expected_df.values.min()
pct_lt5 = (expected_df.values < 5).mean() * 100
pct_lt1 = (expected_df.values < 1).mean() * 100

print(f"min expected={min_exp:.2f}, %cells<5={pct_lt5:.1f}%, %cells<1={pct_lt1:.1f}%, asymptotic p={p_asymp:.4g}")

In [ ]:
import matplotlib.colors as mcolors

# --- Compute adjusted Pearson residuals ---
obs = tab.values
exp = expected

n = obs.sum()
row_sums = obs.sum(axis=1)[:, None]
col_sums = obs.sum(axis=0)[None, :]

# Formula: (O_ij - E_ij) / sqrt(E_ij * (1 - row_sum_i/n) * (1 - col_sum_j/n))
adj_resid = (obs - exp) / np.sqrt(exp * (1 - row_sums/n) * (1 - col_sums/n))
adj_resid_df = pd.DataFrame(adj_resid, index=tab.index, columns=tab.columns)

# Bonferroni correction
alpha = 0.05
n_tests = adj_resid.size
alpha_bonf = alpha / n_tests
crit_val_bonf = norm.ppf(1 - alpha_bonf/2)  # two-sided

sig_mask = np.abs(adj_resid) > crit_val_bonf
enrichment_df = pd.DataFrame(
    np.where(sig_mask, np.sign(adj_resid), 0),  # +1 enriched, -1 depleted, 0 NS
    index=tab.index,
    columns=tab.columns
)

print(f"Chi2={chi2:.2f}, dof={dof}, asymptotic p={p_asymp:.4g}")
print(f"Bonferroni correction: alpha={alpha:.3f}, n_tests={n_tests}, alpha_bonf={alpha_bonf:.3g}, crit_val={crit_val_bonf:.2f}")
#print("\nAdjusted Pearson residuals:")
#display(adj_resid_df)

color_below = "#4575b4"   # For values < -crit_val_bonf (e.g., blue)
color_ns = "#e0e0e0"      # For non-significant values (e.g., light gray)
color_above = "#d73027"   # For values > crit_val_bonf (e.g., red)
continuous_cmap = plt.get_cmap("Reds_r")  # For significant positive values

# Prepare the data
v = adj_resid_df.values
vmin, vmax = float(np.nanmin(v)), float(np.nanmax(v))
crit = float(crit_val_bonf)

# Define colors
color_below = "#D2EEFC"   # light blue
color_ns    = "#e0e0e0"   # gray
color_above_start = "#fcbba1"  # light red
color_above_end   = "#b30000"  # dark red

# Normalized cutoff positions (map data range [vmin, vmax] to [0,1])
p_neg = ( -crit - vmin) / (vmax - vmin)
p_pos = (  crit - vmin) / (vmax - vmin)

# Build segmented colormap
cmap = mcolors.LinearSegmentedColormap.from_list(
    "custom_piecewise",
    [
        (0.0, color_below),    # below -crit
        (p_neg, color_below),
        (p_neg, color_ns),     # non-significant zone
        (p_pos, color_ns),
        (p_pos, color_above_start),
        (1.0, color_above_end) # strongest red
    ]
)

# Plot
g = sns.clustermap(
    adj_resid_df,
    cmap=cmap,
    vmin=vmin, vmax=vmax,
    annot=True, fmt=".2f",
    cbar_kws={"label": "Adjusted Residuals"},
    row_cluster=True, col_cluster=True,
)

# Add cutoff markers to colorbar
cbar = g.ax_heatmap.collections[0].colorbar
for y in (-crit, crit):
    if vmin < y < vmax:
        cbar.ax.axhline(y, color="k", lw=0.8, alpha=0.8)

plt.title("Custom Clustermap: Enrichment Analysis")
plt.show()

pos_mask = adj_resid_df.values > crit_val_bonf

# Compute two-sided p-values from adjusted residuals
pvals = 2 * (1 - norm.cdf(np.abs(adj_resid_df.values)))
pvals = np.clip(pvals, 1e-300, 1)  # ensures no 0, no >1

# Only keep p-values for positively enriched cells, set others to NaN
pvals_pos = np.where(pos_mask, pvals, np.nan)

# Prepare DataFrame for plotting
pvals_pos_df = pd.DataFrame(
    pvals_pos,
    index=adj_resid_df.index,
    columns=adj_resid_df.columns
)

log_pval_pos = -np.log10(pvals_pos_df)
log_pval_pos = log_pval_pos.fillna(0)

# Plot heatmap of -log10(p-values) for positively enriched cells
plt.figure(figsize=(10, 6))
sns.clustermap(
    log_pval_pos,
    annot=True, fmt=".1f",
    cmap="Reds",
    cbar_kws={"label": "-log10(p-value)"},
    linewidths=0.1,
    linecolor='gray',
    mask=pvals_pos_df.isna()
)
plt.title("Heatmap: -log10(p-value) for Positively Enriched Cells")
plt.ylabel("Cluster")
plt.xlabel("mut_cond")
plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'pval_enrichment_heatmap_{("_").join(single_exp)}.pdf'), dpi=300)
plt.show()

## spider plot per cluster

In [ ]:
import math

def make_spider(df, row, color, labels, subplot_index, total_subplots, cluster_0):
    categories = list(df)[0:]
    N = len(categories)

    angles = [n / float(N) * 2 * math.pi for n in range(N)]
    angles += angles[:1]

    n_cols = 4 if N>6 else 3

    plt.subplot(total_subplots // n_cols + (total_subplots % n_cols > 0), n_cols, subplot_index + 1, polar=True)

    plt.xticks(angles[:-1], categories, color='black', size=8)
    plt.tick_params(axis='x', pad=10)

    plt.yticks([0.25, 0.5, 0.75], color="black", size=7)
    plt.ylim(0, 1.0)

    values = df.reset_index().loc[row].values.tolist()[1:]
    values += values[:1]
    linewidht = 3.5 if total_subplots > 1 else 2
    plt.plot(angles, values, color=color, linewidth=3.5, linestyle='solid', label=labels[row])
    plt.fill(angles, values, color=color, alpha=0.6 if total_subplots > 1 else 0.3)

    #title_fontweight = 'bold' if total_subplots > 1 else 'normal'
    plt.title(labels[row], size=11, color=color, y=1.2, fontweight='bold')


# Plot spider charts for each cluster
total_clusters = len(class_proportion)
plt.figure(figsize=(18, 18))

for i, cluster_label in enumerate(class_proportion.index):
    adjusted_cluster_label = cluster_label if cluster_0 else cluster_label - 1
    make_spider(class_proportion, adjusted_cluster_label, color=palette[adjusted_cluster_label], labels=labels, subplot_index=i, total_subplots=total_clusters, cluster_0=cluster_0)
plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'spider_charts_clusters_{("_").join(single_exp)}.pdf'))
plt.show()

# Plot spider charts for each cluster in separate figures
#for i, cluster_label in enumerate(class_proportion.index):
#    plt.figure(figsize=(10, 10))
#    adjusted_cluster_label = cluster_label if cluster_0 else cluster_label - 1
#    make_spider(class_proportion, adjusted_cluster_label, color=palette[adjusted_cluster_label], labels=labels, subplot_index=0, total_subplots=1, cluster_0=cluster_0)
#    plt.savefig(os.path.join(saving_dir, f'spider_chart_single_{adjusted_cluster_label}.pdf'))
#    plt.show()

In [ ]:
from utils.mypalette import palette_mutcond

fig, ax = plt.subplots(figsize=(12, 8))

display_columns = [c.replace('\n', ' ') for c in class_proportion.columns]
class_proportion.columns = display_columns
colors = [palette_mutcond.get(c, '#888888') for c in class_proportion.columns]

class_proportion.plot(kind='bar', stacked=True, ax=ax,
                      color=colors)

ax.set_xlabel('Cluster')
ax.set_ylabel('Proportion')
ax.set_title('Class Proportions in Each Cluster')
ax.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
n_clusters = class_proportion.shape[0]
ncols = 3
nrows = (n_clusters // ncols) + (n_clusters % ncols > 0)

temp_palette = {'CTRL': palette_mutcond['CTRL_A1 low'],
                'VCP': mutation_palette['VCP']}

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)

for idx, (cluster_label, row) in enumerate(class_proportion.iterrows()):
    ax = axes[idx // ncols, idx % ncols]
    #ax.bar(class_proportion.columns, row.values, color=mutation_palette.get(cluster_label, 'black'))
    hue_labels = [col.split('\n')[0] for col in class_proportion.columns]
    
    # 2. Plot with hue mapping to the extracted prefixes
    sns.barplot(
        x=class_proportion.columns, 
        y=row.values, 
        ax=ax, 
        hue=hue_labels, 
        palette=temp_palette,  # Uses your dictionary to map colors
        dodge=False                # Keeps bars centered on the x-ticks
    )
    ax.set_title(f'Cluster {cluster_label}', fontsize=14)
    ax.set_ylabel('fraction')
    y_max = row.values.max()
    ax.set_yticks([0, y_max])
    ax.set_yticklabels([0, f"{y_max:.2f}"])
    #ax.set_ylim(0, 1)
    ax.set_xticklabels(class_proportion.columns, rotation=45, ha='center')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.get_legend().remove()

# Hide unused subplots
for idx in range(n_clusters, nrows * ncols):
    axes[idx // ncols, idx % ncols].axis('off')

plt.tight_layout()
plt.suptitle('Distribution of Biological Classes in Each Cluster', fontsize=16, y=1.04)
plt.savefig(os.path.join(saving_dir, f'cluster_distribution_BIO_{("_").join(single_exp)}.pdf'))
plt.show()

## Shannon entropy

In [ ]:
for label in ['mut_cond']: # Add 'mutation', 'condition' to list if needed
    # --- 1. DATA PREPARATION (Performed Once) ---
    annotations_df['group'] = annotations_df[label]
    
    if label == 'mutation':
        temp_annotations_df = annotations_df[annotations_df['condition'] == 'untreated']
        second_label = 'untreated'
        group_order = mut_order
    elif label == 'condition':
        temp_annotations_df = annotations_df[annotations_df['mutation'] == 'CTRL']
        second_label = 'CTRL'
        group_order = cond_order
    else:
        temp_annotations_df = annotations_df.copy()
        second_label = ' '
        group_order = [g.replace('_', '\n') for g in desired_order]

    # Standardize clusters
    all_cluster_labels = sorted(annotations_df['cluster'].unique())
    
    # Calculate Abundance
    abundance = (
        temp_annotations_df
        .groupby('group')['cluster']
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .reindex(columns=all_cluster_labels, fill_value=0)
    )

    # Reindex rows
    abundance = abundance.reindex([g for g in group_order if g in abundance.index])

    # Initialize entropy DF
    mutcond_entropy_df = pd.DataFrame(columns=['mut_cond', 'entropy'])

    # Grid setup
    n_vals = abundance.shape[0]
    ncols = min(3, n_vals)
    nrows = math.ceil(n_vals / ncols)

    # --- 2. PLOTTING LOOP (Iterate over styles) ---
    for plot_type in ['bar', 'pie']:
        
        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
        
        for idx, (group, row) in enumerate(abundance.iterrows()):
            ax = axes[idx // ncols, idx % ncols]
            p = row.values
            
            # Common Calculation: Shannon Entropy
            entropy_val = (-np.sum(p * np.log(p + 1e-9))) / np.log(len(p)) if len(p) > 1 else 0
            
            # Store entropy only on the first pass (e.g., bar) to avoid duplication in DF
            if plot_type == 'bar':
                mutcond_entropy_df = pd.concat([
                    mutcond_entropy_df, 
                    pd.DataFrame({'mut_cond': group, 'entropy': entropy_val}, index=[0])
                ], ignore_index=True)

            # --- PLOT SPECIFIC LOGIC ---
            if plot_type == 'bar':
                # Bar Plot
                ax.bar(row.index, row.values, color=cl_palette[:len(row)])
                ax.set_ylabel("Proportion")
                ax.set_xticks(range(len(row.index)))
                ax.set_xticklabels(row.index, rotation=45, ha='right')
                y_max = row.values.max()
                ax.set_yticks([0, y_max])
                ax.set_yticklabels([0, f"{y_max:.2f}"])
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                
            elif plot_type == 'pie':
                # Pie Chart
                # Filter zeros for cleaner pies
                mask = row.values > 0
                p_filtered = row.values[mask]
                
                # Ensure colors match the specific clusters active in this slice
                current_colors = [cl_palette[i] for i, val in enumerate(mask) if val]
                
                # Plot Pie Chart
                # Note: When autopct is removed, ax.pie only returns 2 values (wedges, texts)
                wedges, texts = ax.pie(
                    p_filtered, 
                    labels=None,           # Removes the cluster names around the pie
                    colors=current_colors, 
                    startangle=90
                    # autopct is removed to hide percentages
                )

            # Common Title
            ax.set_title(f"{group}\nShannon entropy={entropy_val:.2f}", fontsize=12)

        # Hide unused subplots
        for idx in range(n_vals, nrows * ncols):
            axes[idx // ncols, idx % ncols].axis('off')

        # Common Saving Logic
        plt.suptitle(f"Cluster Proportions ({plot_type}) for {label} {second_label}", fontsize=16)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        
        save_name = f'cluster_proportions_{plot_type}_{label}_{second_label}_{("_").join(single_exp)}.pdf'
        if plot_type == 'pie':
            plt.savefig(os.path.join(saving_dir, save_name))
        plt.show()

In [ ]:
from utils.mypalette import palette_mutcond


mutcond_entropy_df['condition'] = mutcond_entropy_df['mut_cond'].apply(lambda x: x.split('\n')[1])
mutcond_entropy_df['mutation'] = mutcond_entropy_df['mut_cond'].apply(lambda x: x.split('\n')[0])

fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# 1. Draw the boxplot first (the background)
#sns.boxplot(
#    x='condition', 
#    y='entropy', 
#    data=mutcond_entropy_df, 
#    ax=ax, 
#    order=['untreated', 'A1 low', 'A1 high'],
#    color="white",  # Keeping boxes neutral helps the colored lines pop
#    width=0.5,
#    showfliers=False
#)

# 2. Add the lines connecting the mutations
# We iterate over mutations to draw the paths
for mutation in mutcond_entropy_df['mutation'].unique():
    mutation_data = mutcond_entropy_df[mutcond_entropy_df['mutation'] == mutation].sort_values(
        by='condition', 
        key=lambda x: x.map({'untreated': 0, 'A1 low': 1, 'A1 high': 2})
    )
    ax.plot(mutation_data['condition'], mutation_data['entropy'], 
            color='black', alpha=0.4, linewidth=1, zorder=3)

# 3. Overlay the points (stripplot)
sns.stripplot(
    x='condition', 
    y='entropy', 
    data=mutcond_entropy_df, 
    ax=ax, 
    hue='mutation',
    size=12, 
    jitter=False, 
    palette=mutation_palette, 
    order=['untreated', 'A1 low', 'A1 high'],
    zorder=4  # Ensures points stay on top of lines and boxes
)

# Aesthetics
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(title='Mutation', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'entropy_basal_TIC_{("_").join(single_exp)}.pdf'))
plt.show()

In [ ]:
category_proportion = annotations_df.groupby(['mut_cond', 'cluster']).size().unstack(fill_value=0)
category_proportion = category_proportion.div(category_proportion.sum(axis=1), axis=0)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt


cond_proportion = annotations_df.groupby(['mut_cond', 'cluster']).size().unstack(fill_value=0)
cond_proportion = cond_proportion.div(cond_proportion.sum(axis=1), axis=0)


def make_spider_dynamic_ticks(df, row_index, color, title, categories):
    N = len(categories)
    angles = [n / float(N) * 2 * math.pi for n in range(N)]
    angles += angles[:1]
    
    # Get values for this specific plot
    values = df.iloc[row_index].values.tolist()
    local_max = max(values)
    
    # 1. Set the limit to exactly the local maximum value
    y_limit = local_max if local_max > 0 else 0.1 # Fallback to prevent 0 limit errors
    
    # 2. Generate local ticks (e.g., 0.1, 0.2...) strictly below the limit
    ticks = np.arange(0.1, y_limit, 0.1)
    
    # 3. Append the exact max value to the ticks array so the outer line is displayed
    if len(ticks) == 0 or not np.isclose(ticks[-1], y_limit):
        ticks = np.append(ticks, y_limit)
    
    n_cols = 3
    n_rows = math.ceil(len(df) / n_cols)
    ax = plt.subplot(n_rows, n_cols, row_index + 1, polar=True)
    
    # Set the Y-axis limit and ticks dynamically
    ax.set_ylim(0, y_limit)
    ax.set_yticks(ticks)
    # Changed to .2f (or .3f) to accommodate exact max values like 0.35 instead of truncating
    ax.set_yticklabels([f"{t:.2f}" for t in ticks], color="grey", size=7) 
    
    # Spokes
    plt.xticks(angles[:-1], categories, color='black', size=9)
    
    # Data values
    values += values[:1]
    plt.plot(angles, values, color=color, linewidth=2, linestyle='solid')
    plt.fill(angles, values, color=color, alpha=0.4)
    plt.title(title, size=12, color='black', y=1.2, fontweight='bold')

# 3. Plotting
plt.figure(figsize=(16, 12))

binary_palette = {
    'CTRL\nuntreated': condition_palette['untreated'],
    'VCP\nuntreated': mutation_palette['VCP'],
    'CTRL\nA1 low': condition_palette['untreated'],
    'VCP\nA1 low': mutation_palette['VCP'],
    'CTRL\nA1 high': condition_palette['untreated'],
    'VCP\nA1 high': mutation_palette['VCP'],
}

cluster_names = [f"Cluster {c}" for c in cond_proportion.columns]
condition_names = cond_proportion.index.tolist()

for i, cond in enumerate(condition_names):
    make_spider_dynamic_ticks(
        df=cond_proportion, 
        row_index=i, 
        color=binary_palette[cond], 
        title=cond, 
        categories=cluster_names,
    )

plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f"spider_plots_BIO_{('_').join(single_exp)}.pdf"))
plt.show()

In [ ]:
cell_lines = annotations_df['cell_line'].unique()
cluster_labels = annotations_df['cluster'].unique()

entropy_list = []

for cell_line in cell_lines:
    # Get all samples for this cell line
    cl_data = annotations_df[annotations_df['cell_line'] == cell_line]
    cl_data = cl_data[cl_data['condition']=='untreated']
    # Compute cluster label distribution
    cluster_counts = cl_data['cluster'].value_counts().reindex(cluster_labels, fill_value=0)
    cluster_probs = cluster_counts / cluster_counts.sum()

    shannon_ent_norm = (-np.sum(cluster_probs * np.log(cluster_probs + 1e-9))) / np.log(len(cluster_probs))

    # Determine group (CTRL or VCP)
    group = 'CTRL' if 'CTRL' in cl_data['mutation'].iloc[0] else 'VCP'
    entropy_list.append({
        'cell_line': cell_line,
        'entropy': shannon_ent_norm,
        'group': group
    })

UNTR_entropy_df = pd.DataFrame(entropy_list)

# Plot boxplots, separating CTRL and VCP
plt.figure(figsize=(4, 6))
sns.boxplot(x='group', y='entropy', data=UNTR_entropy_df, palette=mutation_palette)
sns.stripplot(x='group', y='entropy', data=UNTR_entropy_df, hue='cell_line', palette=cell_line_palette,
              size=12, jitter=True, alpha=1, edgecolor='black', linewidth=1, legend=False)
plt.title('Shannon Entropy of Cluster Labels per Cell Line')
plt.savefig(os.path.join(saving_dir, f'entropy_cellline_{("_").join(single_exp)}.pdf'))


In [ ]:
cell_lines = annotations_df['cell_line'].unique()
conditions = annotations_df['condition'].unique()
cluster_labels = annotations_df['cluster'].unique()

entropy_list = []

for cell_line in cell_lines:
    # Get all samples for this cell line
    cl_data = annotations_df[annotations_df['cell_line'] == cell_line]
    for condition in conditions:
        # Filter data for the specific condition
        cl_data_cond = cl_data[cl_data['condition'] == condition]
        if cl_data_cond.empty:
            continue
        # Compute cluster label distribution
        cluster_counts = cl_data_cond['cluster'].value_counts().reindex(cluster_labels, fill_value=0)
        cluster_probs = cluster_counts / cluster_counts.sum()

        shannon_ent_norm = (-np.sum(cluster_probs * np.log(cluster_probs + 1e-9))) / np.log(len(cluster_probs))

        group = cl_data_cond['condition'].iloc[0]
        entropy_list.append({
            'cell_line': cell_line,
            'entropy': shannon_ent_norm,
            'group': group
        })

entropy_df = pd.DataFrame(entropy_list)

# Plot boxplots, separating CTRL and VCP
plt.figure(figsize=(4, 6))
sns.boxplot(x='group', y='entropy', data=entropy_df, palette=condition_palette, order=['untreated', 'A1 low', 'A1 high'],
            boxprops=dict(alpha=.3))
sns.stripplot(
    x='group',
    y='entropy',
    data=entropy_df,
    hue='cell_line',
    palette=cell_line_palette,
    size=12,
    jitter=True,
    alpha=1,
    edgecolor='black',
    linewidth=1,
    order=['untreated', 'A1 low', 'A1 high'],
    legend=False
)

# Add lines connecting points for the same cell line
for cell_line, group_df in entropy_df.groupby('cell_line'):
    conds = group_df['group'].values
    group_df = group_df.set_index('group').reindex(['untreated', 'A1 low', 'A1 high']).reset_index()
    plt.plot(
        group_df['group'],
        group_df['entropy'],
        color=cell_line_palette[cell_line],
        alpha=1,
        linewidth=1.5,
        zorder=0
    )
plt.title('Shannon Entropy of Cluster Labels per Cell Line')
plt.savefig(os.path.join(saving_dir, f'entropy_TIC_cellline_{("_").join(single_exp)}.pdf'))
plt.show()
plt.close()

# Entropy vs cosine similarity (UT --> TIC)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

clusters = sorted(annotations_df['cluster'].unique())
cell_lines = annotations_df['cell_line'].unique()
results = []

for cell_line in cell_lines:
    cl_data = annotations_df[annotations_df['cell_line'] == cell_line]
    # Get distributions
    untreated = cl_data[cl_data['condition'] == 'untreated']['cluster'].value_counts().reindex(clusters, fill_value=0)
    a1_low = cl_data[cl_data['condition'] == 'A1 low']['cluster'].value_counts().reindex(clusters, fill_value=0)
    a1_high = cl_data[cl_data['condition'] == 'A1 high']['cluster'].value_counts().reindex(clusters, fill_value=0)
    # Normalize to proportions
    untreated = untreated / untreated.sum() if untreated.sum() > 0 else untreated
    a1_low = a1_low / a1_low.sum() if a1_low.sum() > 0 else a1_low
    a1_high = a1_high / a1_high.sum() if a1_high.sum() > 0 else a1_high
    # Compute correlations
    if untreated.sum() > 0 and a1_low.sum() > 0:
        cos_sim_low = cosine_similarity([untreated.values], [a1_low.values])[0, 0]
    else:
        cos_sim_low = np.nan
    if untreated.sum() > 0 and a1_high.sum() > 0:
        cos_sim_high = cosine_similarity([untreated.values], [a1_high.values])[0, 0]
    else:
        cos_sim_high = np.nan
    group = 'CTRL' if 'CTRL' in cl_data['mutation'].iloc[0] else 'VCP'
    results.append({
        'cell_line': cell_line,
        'corr_untreated_vs_A1_low': cos_sim_low,
        'corr_untreated_vs_A1_high': cos_sim_high,
        'untreated_entropy': UNTR_entropy_df[UNTR_entropy_df['cell_line'] == cell_line]['entropy'].values[0],
        'group': group
    })

plot_df = pd.DataFrame(results)
ctrl_df = plot_df[plot_df['group'] == 'CTRL']
vcp_df = plot_df[plot_df['group'] == 'VCP']

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)

for i, tic in enumerate(['A1_low', 'A1_high']):
    # Plot CTRL
    ax[i].scatter(
        ctrl_df['untreated_entropy'],
        ctrl_df[f'corr_untreated_vs_{tic}'],
        c=[cell_line_palette.get(cl, '#888888') for cl in ctrl_df['cell_line']],
        s=100, edgecolor='black',
    )
    # Plot VCP
    ax[i].scatter(
        vcp_df['untreated_entropy'],
        vcp_df[f'corr_untreated_vs_{tic}'],
        c=[cell_line_palette.get(cl, '#cccccc') for cl in vcp_df['cell_line']],
        s=100, edgecolor='black',
    )

    # Annotate points
    for _, row in ctrl_df.iterrows():
        ax[i].text(
            row['untreated_entropy'],
            row[f'corr_untreated_vs_{tic}'] + 0.015,
            row['cell_line'],
            fontsize=10, ha='right', va='bottom'
        )
    for _, row in vcp_df.iterrows():
        ax[i].text(
            row['untreated_entropy'],
            row[f'corr_untreated_vs_{tic}'] + 0.015,
            row['cell_line'],
            fontsize=10, ha='left', va='bottom'
        )
    

    ax[i].set_title(f'CTRL & VCP: {tic}')
    ax[i].set_xlabel('Basal Entropy')
    ax[i].set_ylabel(f'Cosine similarity (untreated vs {tic})')
    #ax[i].set_xlim(0, max(max_x, vcp_df['untreated_entropy'].max()) + 0.05)
    #ax[i].set_ylim(0, max(max_y, vcp_df[f'corr_untreated_vs_{tic}'].max()) + 0.05)
    #ax[i].legend()

plt.savefig(os.path.join(saving_dir, f'entropy_vs_cosine_sim_{("_").join(single_exp)}.pdf'))
plt.show()

## How spread are bio classes?

In [ ]:
from scipy.spatial.distance import pdist, squareform

# Function to compute the spread of each mut_cond class
def compute_class_spread(emb, labels) -> pd.DataFrame:
    unique_classes = np.unique(labels)
    sep_character = '\n' if '\n' in unique_classes[0] else '_'
    spreads = pd.DataFrame(index=unique_classes, columns=['mutation', 'condition', 'spread'])
    
    for cls in unique_classes:
        class_points = emb[labels == cls]
        if len(class_points) > 1:
            # Compute pairwise distances and take the mean
            pairwise_distances = pdist(class_points, metric='euclidean')
            spread_value = np.mean(pairwise_distances)
        else:
            spread_value = 0

        spreads.loc[cls] = {
            'mutation': cls.split(sep_character)[0],
            'condition': cls.split(sep_character)[1],
            'spread': spread_value
        }
    
    return spreads

mut_cond_labels = annotations_df['mut_cond'].values

# Compute spreads for each mut_cond class
class_spreads = compute_class_spread(raw_emb, mut_cond_labels)

sns.barplot(x='mutation', y='spread', hue='condition', data=class_spreads, palette=condition_palette)
plt.title('Spread of Each mut_cond Class')
plt.xlabel('Mutation')
plt.ylabel('Spread')
plt.legend(title='Condition', bbox_to_anchor=(1.05, 1), loc='upper left')

## Class proportions in clusters

In [ ]:
# Compute class proportions in clusters
class_proportion = annotations_df.groupby(['cluster', 'mut_cond']).size().unstack(fill_value=0)
class_proportion = class_proportion.div(class_proportion.sum(axis=1), axis=0)

# Compute cluster proportions in classes
cluster_proportion = annotations_df.groupby(['mut_cond', 'cluster']).size().unstack(fill_value=0)
cluster_proportion = cluster_proportion.div(cluster_proportion.sum(axis=1), axis=0)

# Plot clustermap for class proportions in clusters
sns.clustermap(class_proportion, cmap='Greys', figsize=(10, 10))
plt.title('Class Proportions in Clusters')
plt.show()

## Plot clustermap for cluster proportions in classes
#sns.clustermap(cluster_proportion, cmap='Greys', figsize=(10, 10), row_cluster=False)
#plt.title('Cluster Proportions in Classes')
#plt.show()

In [ ]:
# Sort the dataframe based on the ordered mut_cond
annotations_df = annotations_df.sort_values('mut_cond').reset_index(drop=True)

# Calculate the proportion of clusters for each mut_cond
cluster_proportion = annotations_df.groupby(['mut_cond', 'cluster']).size().unstack(fill_value=0)
cluster_proportion = cluster_proportion.div(cluster_proportion.sum(axis=1), axis=0)

# Plotting
fig, ax = plt.subplots(figsize=(12, 8))
cluster_proportion.plot(kind='barh', stacked=True, ax=ax, color=palette[:len(cluster_proportion.columns)])

# Set labels, title, and legend
ax.set_xlabel('Proportion')
ax.set_ylabel('Mutation and Condition')
ax.set_title('Proportion of Clusters for Each Mutation and Condition')
ax.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')

# Adjust layout
plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'cluster_proportion_{("_").join(single_exp)}.pdf'))
plt.show()
plt.close()

In [ ]:
# Display proportion of cluster labels in mutation condition classes as a pie chart
fig, axes = plt.subplots(nrows=len(cluster_proportion.index), ncols=1, figsize=(7, 7 * len(cluster_proportion.index)))
if len(cluster_proportion.index) == 1:
    axes = [axes]
for ax, mut_cond in zip(axes, cluster_proportion.index):
    proportions = cluster_proportion.loc[mut_cond]
    labels = [f'Cluster {cl}' for cl in proportions.index]
    colors = palette[:len(proportions)]
    ax.pie(proportions, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140)
    ax.set_title(f'Cluster Proportions in {mut_cond}')
plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'cluster_proportions_piecharts_{("_").join(single_exp)}.pdf'))
plt.show()

In [ ]:
# Plot pie charts of cluster proportions for CTRL and VCP (aggregated over all conditions)
for mut in ['CTRL', 'VCP']:
    # Select rows for this mutation (all conditions)
    rows = [idx for idx in cluster_proportion.index if idx.startswith(mut)]
    # Aggregate proportions over all conditions for this mutation
    agg_props = cluster_proportion.loc[rows].sum()
    agg_props = agg_props / agg_props.sum()  # Normalize to sum to 1
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(agg_props, labels=[f'Cluster {cl}' for cl in agg_props.index], colors=palette[:len(agg_props)], autopct='', startangle=140)
    ax.set_title(f'Cluster Proportions in {mut}', fontsize=16)
    plt.tight_layout()
    plt.savefig(os.path.join(saving_dir, f'{mut}_cluster_piechart_{("_").join(single_exp)}.pdf'))
    plt.show()

In [ ]:
# Filter annotations_df for CTRL mutation
ctrl_df = annotations_df[annotations_df['mutation'] == 'CTRL']

# Count proportions of each condition label
condition_counts = ctrl_df['condition'].value_counts()
condition_proportions = condition_counts / condition_counts.sum()

# Get colors for each condition from condition_palette
condition_labels = condition_proportions.index.tolist()
colors = [condition_palette.get(cond, '#888888') for cond in condition_labels]

# Plot pie chart
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(condition_proportions, labels=condition_labels, autopct='%1.1f%%', colors=colors, startangle=140)
ax.set_title('Proportions of Condition Labels in CTRL Mutation')
plt.tight_layout()
plt.savefig(os.path.join(saving_dir, f'CTRL_condition_piechart_{("_").join(single_exp)}.pdf'))
plt.show()

In [ ]:
# Plotting
fig, ax = plt.subplots(figsize=(10, 8))
class_proportion.plot(kind='bar', stacked=True, ax=ax, colormap='rainbow')

# Set labels, title, and legend
ax.set_xlabel('Cluster')
ax.set_ylabel('Proportion')
ax.set_title('Proportion of Classes in Each Cluster')
ax.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()
plt.close()

In [ ]:
import random

# Function to display images
def display_images(img_codes):
    images = [Image.open(img_code) for img_code in img_codes]
    fig, axes = plt.subplots(7, 7, figsize=(15, 15))
    for ax, img in zip(axes.flatten(), images):
        ax.imshow(img)
        ax.axis('off')
    plt.suptitle(f'Cluster {cluster_label}', fontsize=20)
    plt.tight_layout()
    plt.show()

# Select 49 random images for each cluster
for cluster_label in sorted(annotations_df['cluster'].unique()):
    cluster_images = annotations_df[annotations_df['cluster'] == cluster_label]['img_code'].tolist()
    if len(cluster_images) > 49:
        cluster_images = random.sample(cluster_images, 49)
    display_images(cluster_images)

In [ ]:
import random

# Function to display images
def display_images_row(img_codes, cluster_label, ax, adjust):
    images = [Image.open(img_code) for img_code in img_codes]
    for i, img in enumerate(images):
        ax[i].imshow(img)
        ax[i].axis('off')
    adjust = 0 if cluster_0 else 1
    ax[0].set_title(f'Cluster {cluster_label+adjust}', fontsize=20, color=cl_palette[cluster_label], fontweight='bold')

n_examples = 20

fig, axes = plt.subplots(nrows=len(annotations_df['cluster'].unique()), ncols=n_examples, figsize=(2*n_examples, 2 * len(annotations_df['cluster'].unique())))
for row, cluster_label in enumerate(sorted(annotations_df['cluster'].unique())):
    cluster_images = annotations_df[annotations_df['cluster'] == cluster_label]['img_code'].tolist()
    if len(cluster_images) > n_examples:
        cluster_images = random.sample(cluster_images, n_examples)
    adjusted_cluster_label = cluster_label if cluster_0 else cluster_label - 1
    display_images_row(cluster_images, adjusted_cluster_label, axes[row], adjust=cluster_0)

plt.tight_layout()
plt.show()

# How well clusters are separated from each other?

In [ ]:
# Compute silhouette score
silhouette_avg = silhouette_score(emb_to_analyze, best_labels)
print("Silhouette Score: ", silhouette_avg)

# Compute Davies-Bouldin index
davies_bouldin_avg = davies_bouldin_score(emb_to_analyze, best_labels)
print("Davies-Bouldin Index: ", davies_bouldin_avg)

# Compute Calinski-Harabasz index
calinski_harabasz_avg = calinski_harabasz_score(emb_to_analyze, best_labels)
print("Calinski-Harabasz Index: ", calinski_harabasz_avg)

# How homogeneous are these clusters? And how spread?

In [ ]:
from utils.clustering_analysis import compute_cluster_entropies, compute_cluster_stats

centroids, spreads = compute_cluster_stats(emb_to_analyze, best_labels)
entropy_dict = compute_cluster_entropies(emb_to_analyze, best_labels, categorical_labels)

In [ ]:
# Prepare data for plotting
entropy_df = pd.DataFrame.from_dict(entropy_dict, orient='index', columns=['entropy'])
entropy_df.reset_index(inplace=True)
entropy_df.rename(columns={'index': 'cluster_cat'}, inplace=True)
entropy_df['cluster'] = entropy_df['cluster_cat'].apply(lambda x: str(x).split('_')[0])
entropy_df['type'] = entropy_df['cluster_cat'].apply(lambda x: 'cat' if 'cat' in str(x) else 'emb')

# Plotting
fig, ax = plt.subplots(figsize=(15, 6))
sns.barplot(x='cluster', y='entropy', data=entropy_df[entropy_df['type']=='cat'], ax=ax, palette=cl_palette)

# Set labels, title, and legend
ax.set_xlabel('Cluster')
ax.set_ylabel('Entropy')
ax.set_title('Entropy of Clusters')
plt.tight_layout()
plt.show()


# Prepare data for plotting
spreads_df = pd.DataFrame([(k, v) for k, values in spreads.items() for v in values], columns=['cluster', 'spread'])
spreads_df = spreads_df.explode('spread')
spreads_df.reset_index(inplace=True)
#spreads_df.rename(columns={'index': 'cluster'}, inplace=True)

fig, ax = plt.subplots()
sns.boxplot(x='cluster', y='spread', data=spreads_df, palette='tab20', ax=ax)
ax.set_xlabel('Cluster')
ax.set_ylabel('Spread')
ax.set_title('Boxplot of Spread by Cluster')
plt.show()

# Which clusters are more similar to each other?

In [ ]:
from scipy.stats import wasserstein_distance

# Function to compute Wasserstein distances between pairs of clusters
def compute_wasserstein_distances(emb, cl_labels):
    unique_labels = np.unique(cl_labels)
    distances = {}
    
    for i, label1 in enumerate(unique_labels):
        for label2 in unique_labels[i+1:]:
            cluster1_points = emb[cl_labels == label1]
            cluster2_points = emb[cl_labels == label2]
            distance = wasserstein_distance(cluster1_points.flatten(), cluster2_points.flatten())
            distances[(label1, label2)] = distance
    
    distances_df = pd.DataFrame([(label1, label2, dist) for (label1, label2), dist in distances.items()],
                                columns=['label1', 'label2', 'distance'])
    return distances_df

# Compute Wasserstein distances
wasserstein_distances = compute_wasserstein_distances(emb_to_analyze, best_labels)
# Create a pivot table for the clustermap
wasserstein_pivot = wasserstein_distances.pivot(index='label1', columns='label2', values='distance')
for i in wasserstein_pivot.index:
    for j in wasserstein_pivot.columns:
        if pd.isna(wasserstein_pivot.at[i, j]):
            wasserstein_pivot.at[i, j] = wasserstein_pivot.at[j, i]

# Fill NaN values with 0 (since distance between the same clusters is 0)
wasserstein_pivot = wasserstein_pivot.fillna(0)

In [ ]:
# Plot the clustermap
sns.clustermap(wasserstein_pivot, cmap='Greys', figsize=(10, 10))
plt.title('Wasserstein Distances Between Clusters')
plt.show()

In [ ]:
for cluster in wasserstein_pivot.index:
    closest_clusters = wasserstein_pivot.loc[cluster].replace(0.0, np.nan).nsmallest(2)
    print(f"Cluster {cluster}:")
    for closest_cluster, distance in closest_clusters.items():
        print(f"  Closest cluster: {closest_cluster}, Distance: {distance}")

In [ ]:
from scipy.spatial.distance import jensenshannon

# Function to compute Jensen-Shannon divergences between pairs of clusters based on categorical labels
def compute_jensenshannon(cl_labels, cat_labels):

    unique_labels = np.unique(cl_labels)
    distances = {}
    label_to_int = {label: idx for idx, label in enumerate(np.unique(cat_labels))}
    int_labels = np.array([label_to_int[label] for label in cat_labels])
    
    for i, label1 in enumerate(unique_labels):
        for label2 in unique_labels[i+1:]:
            # Compute Shannon entropy for each cluster based on categorical labels
            cat_points1 = int_labels[cl_labels == label1]
            cat_points2 = int_labels[cl_labels == label2]
            cat_probabilities1 = np.bincount(cat_points1, minlength=6) / len(cat_points1)
            cat_probabilities2 = np.bincount(cat_points2, minlength=6) / len(cat_points2)
            # Compute Jensen-Shannon divergence
            distance = jensenshannon(cat_probabilities1, cat_probabilities2)
            distances[(label1, label2)] = distance
    
    distances_df = pd.DataFrame([(label1, label2, dist) for (label1, label2), dist in distances.items()],
                                columns=['label1', 'label2', 'distance'])
    return distances_df

# Compute Jensen-Shannon divergences
jensenshannon_distances = compute_jensenshannon(best_labels, categorical_labels)

# Create a pivot table for the clustermap
jensenshannon_pivot = jensenshannon_distances.pivot(index='label1', columns='label2', values='distance')
for i in jensenshannon_pivot.index:
    for j in jensenshannon_pivot.columns:
        if pd.isna(jensenshannon_pivot.at[i, j]):
            jensenshannon_pivot.at[i, j] = jensenshannon_pivot.at[j, i]

# Fill NaN values with 0 (since distance between the same clusters is 0)
jensenshannon_pivot = jensenshannon_pivot.fillna(0)

# Plot the clustermap
sns.clustermap(jensenshannon_pivot, cmap='Greys', figsize=(10, 10))
plt.title('Jensen-Shannon Divergences Between Clusters')
plt.show()


for cluster in jensenshannon_pivot.index:
    closest_clusters = jensenshannon_pivot.loc[cluster].replace(0.0, np.nan).nsmallest(2)
    print(f"Cluster {cluster}:")
    for closest_cluster, distance in closest_clusters.items():
        print(f"  Closest cluster: {closest_cluster}, Jensen-Shannon Divergence: {distance}")

In [ ]:
from scipy.stats import chi2_contingency

# Create a contingency table
contingency_table = pd.crosstab(annotations_df['mut_cond'], annotations_df['cluster'])

# Create a function to compute pairwise chi-square tests
def pairwise_chi_square_tests(contingency_table):
    results = []
    clusters = contingency_table.columns
    for i in range(len(clusters)):
        for j in range(i + 1, len(clusters)):
            cluster1 = clusters[i]
            cluster2 = clusters[j]
            sub_table = contingency_table[[cluster1, cluster2]]
            sub_table += 0.1
            chi2, p, dof, expected = chi2_contingency(sub_table)
            results.append({
                'Cluster1': cluster1,
                'Cluster2': cluster2,
                'Chi-square': chi2,
                'P-value': p,
                'Degrees of freedom': dof
            })
    return pd.DataFrame(results)

# Compute pairwise chi-square tests
pairwise_results = pairwise_chi_square_tests(contingency_table)

# Display the results
print(pairwise_results)